# 2110413 COMPUTER SECURITY

## **Homework 01:** Hacking Password

- **Author:** Worralop Srichainont
- **Year:** 2026 (Semester 1)

# Dependencies

**Required libraries**

- `hashlib`: Python standard library, provides MD5 / SHA-1 / SHA-256 / SHA-512 hashing.
- `bcrypt`: third-party library for password-specific hashing.

In [39]:
%pip install bcrypt

In [40]:
import hashlib
import bcrypt

import time
import itertools
import urllib.request

**Required files**

- Common password dictionary: [`10k-most-common.txt`](https://github.com/danielmiessler/SecLists/blob/master/Passwords/Common-Credentials/10k-most-common.txt)

In [41]:
COMMON_PASSWORDS_URL = "https://raw.githubusercontent.com/danielmiessler/SecLists/refs/heads/master/Passwords/Common-Credentials/10k-most-common.txt"
COMMON_PASSWORDS_PATH = "10k-most-common.txt"

COMMON_PASSWORDS = []

In [42]:
# Download the file
urllib.request.urlretrieve(COMMON_PASSWORDS_URL, COMMON_PASSWORDS_PATH)

# Store each password into a list
with open(COMMON_PASSWORDS_PATH, encoding="utf-8") as file:
    for line in file:
        COMMON_PASSWORDS.append(line.strip())

In [43]:
# Display the stored list
print("Common Password Dictionary")
print(f"- Quantity: {len(COMMON_PASSWORDS)} items")
print(f"- First 5 items: {COMMON_PASSWORDS[:5]}")

Common Password Dictionary
- Quantity: 10000 items
- First 5 items: ['password', '123456', '12345678', '1234', 'qwerty']


# Exercise 01

**Objective**: Understand how attackers use pre-built word lists (dictionaries) to crack hashes of common passwords.

**Scenario**: You have discovered a SHA-1 hash in a compromised system:

```
d54cc1fe76f5186380a0939d2fc1723c44e8a5f7
```

You suspect the password is a simple, common word, possibly with some character substitutions.


**Task**: Write a Python program that reads a list of words, applies common substitutions, hashes the result, and checks if it matches the target hash.
Note that you might want to include substitution in your code (lowercase, uppercase, number for letter [`o` -> `0` , `l` -> `1`, `i` -> `1`]).

## Solution

Define a target SHA-1 password hash

In [44]:
TARGET_HASH = "d54cc1fe76f5186380a0939d2fc1723c44e8a5f7"

Define a letter substitution which includes lowercase, uppercase and number that looks similar.

In [45]:
LETTER_SUBSTITUTIONS = {
    "a": ["a", "A", "4"],
    "b": ["b", "B", "6", "8"],
    "c": ["c", "C"],
    "d": ["d", "D"],
    "e": ["e", "E", "3"],
    "f": ["f", "F"],
    "g": ["g", "G", "6", "9"],
    "h": ["h", "H"],
    "i": ["i", "I", "1"],
    "j": ["j", "J"],
    "k": ["k", "K"],
    "l": ["l", "L", "1"],
    "m": ["m", "M"],
    "n": ["n", "N"],
    "o": ["o", "O", "0"],
    "p": ["p", "P"],
    "q": ["q", "Q"],
    "r": ["r", "R"],
    "s": ["s", "S", "5"],
    "t": ["t", "T", "7"],
    "u": ["u", "U"],
    "v": ["v", "V"],
    "w": ["w", "W"],
    "x": ["x", "X"],
    "y": ["y", "Y"],
    "z": ["z", "Z", "2"],
    "0": ["0", "o", "O"],
    "1": ["1", "i", "I", "l"],
    "2": ["2", "z", "Z"],
    "3": ["3", "e", "E"],
    "4": ["4", "a", "A"],
    "5": ["5", "s", "S"],
    "6": ["6", "b", "G"],
    "7": ["7", "t", "T"],
    "8": ["8", "B"],
    "9": ["9", "g"],
}

Create these functions inside the class

**Function 01**: Generates a variant of the password
- **Lowercase/Uppercase**: Generate a combination of lowercase and uppercase (e.g., `hello` -> [`hello`, `Hello`, `helLo`])
- **Number for Letter**: Replace a English letter with a similar number (e.g., `hello` -> [`h3ll0`, `he11o`, `h3110`])

**Function 02**: SHA-1 hash cracker
- Iterate every password in common password dictionary
- Each password, generates variant of it.
- Check each variant whether it matches the target hash and store the result if matches

In [46]:
class Exercise01:
    @staticmethod
    def get_password_from_sha1_hash(
        password_list: list[str], target_hash: str
    ) -> list[str]:
        """
        Tries every password in the list and return all password variants that matches the SHA1 hash

        Args:
            password_list (list[str]): list of attempt password
            target_hash (str): target SHA1 hash

        Returns:
            list[str]: list of passwords variants that matches the
        """

        match_variants = []
        for password in password_list:
            match_variants.extend(
                Exercise01.__get_match_variants(password, target_hash)
            )
        return match_variants

    @staticmethod
    def __generate_password_variants(password: str) -> list[str]:
        """
        Generate a password uppercase/lowercase and number combination

        Args:
            password (str): original password

        Returns:
            list[str]: list of password combination
        """

        # Get the substitution list in for each character in the password
        password = password.lower()
        options = [LETTER_SUBSTITUTIONS.get(char, [char]) for char in password]

        # Generate all permutation
        permutations = ["".join(comb) for comb in itertools.product(*options)]
        return permutations

    @staticmethod
    def __get_match_variants(password: str, target_hash: str) -> list[str]:
        """
        Tires each variant of a single password and return those that matches the target SHA1 hash

        Args:
            password (str): the original password
            target_hash (str): target SHA1 hash

        Returns:
            list[str]: list of password variants that matches the target hash
        """

        match_variants = []
        password_variants = Exercise01.__generate_password_variants(password)

        for variant in password_variants:
            if (
                hashlib.sha1(variant.encode()).hexdigest().lower()
                == target_hash.lower()
            ):
                match_variants.append(variant)
        return match_variants

Run the experiment

In [47]:
start_time = time.time()
result = Exercise01.get_password_from_sha1_hash(
    password_list=COMMON_PASSWORDS, target_hash=TARGET_HASH
)
elapsed_time = time.time() - start_time

In [48]:
print( "================= EXPERIMENT REPORT ==================")
print(f"SHA-1 hash:   {TARGET_HASH}")
print(f"Elapsed time: {elapsed_time:.2f} s")
print(f"Result ({len(result)} password)")
for item in result:
    print(f"- {item}")
print( "======================================================")

================= EXPERIMENT REPORT ==================
SHA-1 hash:   d54cc1fe76f5186380a0939d2fc1723c44e8a5f7
Elapsed time: 26.81 s
Result (1 password)
- ThaiLanD


# Exercise 02

## Problem

**Objective**: To understand why modern password hashing algorithms like bcrypt are more secure than older ones like MD5 and SHA-1.

**Task**: Design and run an experiment to measure how many hashes each algorithm can compute in a fixed amount of time. The code must test at least `MD5`, `SHA-1`, and `bcrypt`.
(You may also try additional algorithms like `SHA256`, `SHA512`, `scrypt`, `Argon2`.)

**Hint**: Use `time` function in python.

In [ ]:
class Exercise02:
    @staticmethod
    def run_hashlib(algo_name, duration=3):
        count = 0
        start = time.time()
        while time.time() - start < duration:
            hashlib.new(algo_name, str(count).encode()).hexdigest()
            count += 1
        return count / (time.time() - start)

    @staticmethod
    def benchmark_bcrypt(duration=1.5, rounds=12):
        count = 0
        start = time.time()
        while time.time() - start < duration:
            bcrypt.hashpw(str(count).encode(), bcrypt.gensalt(rounds=rounds))
            count += 1
        return count / (time.time() - start)